# T02: Detection Calibration Notebook

This notebook reproduces a detection run end-to-end from a sample input through intake, normalization, KGW detection, and the resulting `DetectionResult`.

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start=Path.cwd()):
    """Find the checkout root from any working directory inside the repository."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Repository root not found; launch from within the checkout.")


REPO_ROOT = find_repo_root()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from watermark_lab.core.contracts import DetectionConfig
from watermark_lab.run_engine.detect import DetectionService
from watermark_lab.run_engine.intake import compute_sha256, normalize_text, read_source

sample_text = (
    "The quick brown fox jumps over the lazy dog. The quick brown fox jumps over the lazy dog "
    "again. The quick brown fox jumps over the lazy dog one more time."
)
text, media_type, source_path = read_source(sample_text)
normalized = normalize_text(text)

config = DetectionConfig(
    method_id="kgw",
    tokenizer_id="gpt2",
    key_id="test-kgw-v1",
    threshold=4.0,
    options={"gamma": 0.25, "context_width": 1},
)

result = DetectionService().detect(normalized, config)

report = {
    "input_sha256": compute_sha256(text),
    "normalized_sha256": compute_sha256(normalized),
    "method_id": result.method_id,
    "compatibility": result.compatibility,
    "decision": result.decision,
    "score": result.score,
    "p_value": result.p_value,
    "token_count": result.token_count,
    "estimated_fpr": result.estimated_fpr,
    "warnings": list(result.warnings),
}
report

## Run manifest

The run manifest below is validated against `schemas/run-manifest.schema.json`.

In [ ]:
import json
from datetime import datetime, timezone
from jsonschema import validate

manifest_path = REPO_ROOT / "schemas" / "run-manifest.schema.json"
schema = json.loads(manifest_path.read_text(encoding="utf-8"))

manifest = {
    "run_id": "0192b5c0-0000-7000-8000-000000000002",
    "parent_run_id": None,
    "run_type": "detect",
    "method_id": "kgw",
    "compatibility": "experimental",
    "input_sha256": report["input_sha256"],
    "normalized_sha256": report["normalized_sha256"],
    "created_at": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "software_version": "0.1.0",
    "git_commit": None,
    "key_id": "test-kgw-v1",
    "config": {
        "method_id": "kgw",
        "tokenizer_id": "gpt2",
        "key_id": "test-kgw-v1",
        "threshold": 4.0,
        "options": {"gamma": 0.25, "context_width": 1},
    },
    "artifacts": [{"path": "report.json", "role": "report"}],
}

validate(instance=manifest, schema=schema)
print("Run manifest is valid.")

## Summary

- The Detect workflow is wired end-to-end from intake to `DetectionResult`.
- The KGW detector returns allowed decisions only.
- `estimated_fpr` is `None` until the calibration corpus is built.
- No real watermark key or API key appears in the notebook.